In [2]:
from pathlib import Path
import importlib
import shutil
import sys

import numpy as np
import pandas as pd
import torch
from torch import nn

# --------------------------------------------------
# 1. Remove the unused import, preserving a backup.
# --------------------------------------------------
REPO = Path.home() / "tva_flood/GraphRivers"
model_path = REPO / "gwnetmodel.py"
backup_path = REPO / "gwnetmodel.py.before_import_fix.bak"

assert model_path.is_file(), f"Model file not found: {model_path}"

text = model_path.read_text()
unused_import = "from util_gtnet import Time2Vec"

if unused_import in text:
    if not backup_path.exists():
        shutil.copy2(model_path, backup_path)

    model_path.write_text(
        text.replace(
            unused_import,
            "# Unused Time2Vec import removed."
        )
    )
    print("Removed unused Time2Vec import.")
else:
    print("Unused import already removed.")

# Import from this repository, refreshing any cached module.
sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
sys.modules.pop("gwnetmodel", None)
sys.modules.pop("utils_wnet", None)

from gwnetmodel import GWNet
from utils_wnet import load_adj

# --------------------------------------------------
# 2. Load the existing pilot graph.
# --------------------------------------------------
ROOT = Path("/data/tva_data")
PILOT = ROOT / "processed/pilot_graph/rosn7"

nodes = pd.read_parquet(PILOT / "nodes.parquet")
nodes = nodes.sort_values("node_index").reset_index(drop=True)
edges = pd.read_parquet(PILOT / "edges.parquet")
n = len(nodes)

assert n > 0
assert nodes.comid.is_unique
assert np.array_equal(nodes.node_index.values, np.arange(n))
assert edges.source.between(0, n - 1).all()
assert edges.target.between(0, n - 1).all()

# Check that edge indices refer to the expected COMIDs.
ids = nodes.comid.to_numpy(dtype=np.int64)
source = edges.source.to_numpy(dtype=np.int64)
target = edges.target.to_numpy(dtype=np.int64)

assert np.array_equal(
    ids[source], edges.fromcomid.to_numpy(dtype=np.int64)
)
assert np.array_equal(
    ids[target], edges.tocomid.to_numpy(dtype=np.int64)
)

# Sender rows and receiver columns, matching existing nconv.
adjacency = np.eye(n, dtype=np.float32)
adjacency[source, target] = 1.0

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.manual_seed(42)
if device.type == "cuda":
    torch.cuda.manual_seed_all(42)

supports = [
    torch.as_tensor(
        np.asarray(a), dtype=torch.float32, device=device
    )
    for a in load_adj(adjacency, "doubletransition")
]

# --------------------------------------------------
# 3. Instantiate the existing GraphRivers model.
# --------------------------------------------------
model = GWNet(
    device=device,
    num_nodes=n,
    supports=supports,
    addaptadj=False,
    in_dim=16,       # Eight weather values + eight masks.
    out_dim=40,
    blocks=4,
    layers=5,       # Receptive field = 125 steps.
    residual_channels=8,
    dilation_channels=8,
    skip_channels=32,
    end_channels=64,
)

# Correct legacy Conv1d layers on this instance only.
# These changes do not overwrite the model source.
for name in ("gate_convs", "residual_convs", "skip_convs"):
    corrected = []

    for layer in getattr(model, name):
        if isinstance(layer, nn.Conv1d):
            replacement = nn.Conv2d(
                in_channels=layer.in_channels,
                out_channels=layer.out_channels,
                kernel_size=layer.kernel_size,
                stride=layer.stride,
                padding=layer.padding,
                dilation=layer.dilation,
                groups=layer.groups,
                bias=layer.bias is not None,
                padding_mode=layer.padding_mode,
            )
            replacement.load_state_dict(layer.state_dict())
            corrected.append(replacement)
        else:
            corrected.append(layer)

    setattr(model, name, nn.ModuleList(corrected))

model = model.to(device)
model.train()

assert model.receptive_field >= 120

# --------------------------------------------------
# 4. Synthetic forward pass with missing-input masks.
# --------------------------------------------------
weather = torch.randn(2, 120, n, 8, device=device)
input_mask = torch.rand_like(weather) > 0.1

weather = torch.where(
    input_mask, weather, torch.zeros_like(weather)
)
inputs = torch.cat(
    [weather, input_mask.float()], dim=-1
)

model.zero_grad(set_to_none=True)
output = model(inputs)

assert output.shape == (2, 1, n, 40), (
    f"Unexpected model output: {tuple(output.shape)}"
)

# Convert to [batch, forecast step, node, discharge variable].
forecast = output[:, 0].permute(0, 2, 1).unsqueeze(-1)
assert forecast.shape == (2, 40, n, 1)
assert torch.isfinite(forecast).all()

# --------------------------------------------------
# 5. Masked loss and backward pass.
# --------------------------------------------------
targets = torch.randn_like(forecast)
target_mask = torch.rand_like(targets) > 0.1
assert target_mask.any()

targets[~target_mask] = float("nan")

# Select available targets before computing the loss.
loss = (
    forecast[target_mask] - targets[target_mask]
).square().mean()

assert torch.isfinite(loss)
loss.backward()

gradients = [
    p.grad for p in model.parameters() if p.grad is not None
]
assert gradients, "No gradients were produced."
assert all(torch.isfinite(g).all() for g in gradients)

print("\nCOMPATIBILITY CHECK PASSED")
print("Device:", device)
print("Nodes / edges:", n, len(edges))
print("Receptive field:", model.receptive_field)
print("Input shape:", tuple(inputs.shape))
print("Forecast shape:", tuple(forecast.shape))
print("Available synthetic targets:", int(target_mask.sum().item()))
print("Synthetic loss:", float(loss.detach().cpu()))
print("Forward pass, masked loss, and backward pass: PASSED")

Removed unused Time2Vec import.

COMPATIBILITY CHECK PASSED
Device: cuda
Nodes / edges: 81 80
Receptive field: 125
Input shape: (2, 120, 81, 16)
Forecast shape: (2, 40, 81, 1)
Available synthetic targets: 5847
Synthetic loss: 1.037392497062683
Forward pass, masked loss, and backward pass: PASSED


In [3]:
from pathlib import Path
from collections import OrderedDict
import importlib
import shutil
import sys

import numpy as np
import pandas as pd
import xarray as xr
import torch
from torch.utils.data import Dataset, DataLoader

ROOT = Path("/data/tva_data")
REPO = Path.home() / "tva_flood/GraphRivers"
PILOT = ROOT / "processed/pilot_graph/rosn7"

VARIABLES = [
    "APCP_surface", "TMP_2maboveground", "SPFH_2maboveground",
    "PRES_surface", "DSWRF_surface", "DLWRF_surface",
    "UGRD_10maboveground", "VGRD_10maboveground",
]

START = pd.Timestamp("2018-01-01")
STOP = pd.Timestamp("2018-07-01")   # Exclusive.
HISTORY, HORIZON = 120, 40

# --------------------------------------------------
# 1. Permanently fix the existing model, with backup.
# --------------------------------------------------
path = REPO / "gwnetmodel.py"
backup = REPO / "gwnetmodel.py.before_forecast_fix.bak"
assert path.is_file(), path

text = path.read_text()
updated = text.replace(
    "from util_gtnet import Time2Vec",
    "# Unused Time2Vec import removed."
).replace("nn.Conv1d(", "nn.Conv2d(")

if updated != text:
    if not backup.exists():
        shutil.copy2(path, backup)
    path.write_text(updated)
    print("Saved model convolution fixes; backup:", backup)

sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
sys.modules.pop("gwnetmodel", None)
sys.modules.pop("utils_wnet", None)

from gwnetmodel import GWNet
from utils_wnet import load_adj

# --------------------------------------------------
# 2. Load pilot node ordering and connectivity.
# --------------------------------------------------
nodes = pd.read_parquet(PILOT / "nodes.parquet")
nodes = nodes.sort_values("node_index").reset_index(drop=True)
edges = pd.read_parquet(PILOT / "edges.parquet")

IDS = nodes.comid.to_numpy(dtype=np.int64)
N = len(IDS)

assert nodes.comid.is_unique
assert np.array_equal(nodes.node_index.values, np.arange(N))

source = edges.source.to_numpy(dtype=np.int64)
target = edges.target.to_numpy(dtype=np.int64)
assert ((source >= 0) & (source < N)).all()
assert ((target >= 0) & (target < N)).all()
assert np.array_equal(IDS[source], edges.fromcomid.values)
assert np.array_equal(IDS[target], edges.tocomid.values)

# --------------------------------------------------
# 3. Monthly reader: retain only the selected 81 nodes.
# --------------------------------------------------
class PilotMonthlyReader:
    def __init__(self, root, ids, start, stop, cache_months=2):
        self.ids = ids
        self.cache = OrderedDict()
        self.cache_months = cache_months
        self.months = pd.date_range(
            start, stop, freq="MS", inclusive="left"
        )
        self.paths = []
        self.positions = []
        self.offsets = [0]
        time_parts = []

        for first in self.months:
            last = first + pd.offsets.MonthBegin(1)
            stamp = f"{first:%Y%m%dT%H}_{last:%Y%m%dT%H}"
            path = (
                root / "processed/six_hour_nodes" / str(first.year)
                / f"tennessee_{stamp}_six_hour_nodes.nc"
            )
            assert path.is_file(), f"Missing monthly file: {path}"

            with xr.open_dataset(path, cache=False) as ds:
                times = pd.DatetimeIndex(ds.time.values)
                expected = pd.date_range(
                    first, last, freq="6h", inclusive="left"
                )
                assert times.equals(expected), f"Time mismatch: {path}"
                index = pd.Index(ds.feature_id.values)
                assert index.is_unique
                positions = index.get_indexer(ids)
                assert (positions >= 0).all(), f"Missing COMID: {path}"
                assert np.array_equal(index.values[positions], ids)

                for name in VARIABLES + ["streamflow"]:
                    assert ds[name].dims == ("time", "feature_id")
                    assert ds[name + "_valid_hours"].dims == (
                        "time", "feature_id"
                    )

            self.paths.append(path)
            self.positions.append(positions)
            time_parts.append(times.values)
            self.offsets.append(self.offsets[-1] + len(times))

        self.offsets = np.asarray(self.offsets)
        self.times = pd.DatetimeIndex(np.concatenate(time_parts))
        expected = pd.date_range(start, stop, freq="6h", inclusive="left")
        assert self.times.equals(expected), "Archive has a time gap."

    def month(self, number):
        if number in self.cache:
            self.cache.move_to_end(number)
            return self.cache[number]

        with xr.open_dataset(
            self.paths[number], cache=False, engine="netcdf4"
        ) as ds:
            selected = ds.isel(feature_id=self.positions[number])

            weather = np.stack(
                [selected[v].values for v in VARIABLES], axis=-1
            ).astype(np.float32)

            weather_counts = np.stack(
                [selected[v + "_valid_hours"].values for v in VARIABLES],
                axis=-1
            )
            weather_mask = (
                np.isfinite(weather) & (weather_counts == 6)
            )

            flow = selected.streamflow.values.astype(np.float32)
            flow_counts = selected.streamflow_valid_hours.values
            flow_mask = np.isfinite(flow) & (flow_counts == 6)

            assert not (flow[flow_mask] < 0).any(), (
                f"Negative valid discharge: {self.paths[number]}"
            )

        result = (weather, weather_mask, flow, flow_mask)
        self.cache[number] = result
        self.cache.move_to_end(number)
        while len(self.cache) > self.cache_months:
            self.cache.popitem(last=False)
        return result

    def window(self, start_index, stop_index):
        assert 0 <= start_index < stop_index <= len(self.times)
        parts = [[], [], [], []]
        cursor = start_index

        while cursor < stop_index:
            month = int(
                np.searchsorted(self.offsets, cursor, side="right") - 1
            )
            end = min(stop_index, int(self.offsets[month + 1]))
            left = cursor - self.offsets[month]
            right = end - self.offsets[month]

            for destination, values in zip(parts, self.month(month)):
                destination.append(values[left:right])
            cursor = end

        return tuple(np.concatenate(p, axis=0) for p in parts)

reader = PilotMonthlyReader(ROOT, IDS, START, STOP)

# --------------------------------------------------
# 4. Fit normalization on this training pilot only.
# Weather: shared per-variable mean/std across nodes.
# Discharge: shared mean/std after log1p transformation.
# --------------------------------------------------
count = np.zeros(9, dtype=np.int64)
total = np.zeros(9, dtype=np.float64)
squares = np.zeros(9, dtype=np.float64)

for month in range(len(reader.paths)):
    weather, weather_mask, flow, flow_mask = reader.month(month)
    valid_flow = np.log1p(flow[flow_mask].astype(np.float64))

    for feature in range(9):
        values = (
            weather[..., feature][weather_mask[..., feature]]
            .astype(np.float64)
            if feature < 8 else valid_flow
        )
        count[feature] += values.size
        total[feature] += values.sum()
        squares[feature] += np.square(values).sum()

assert (count > 0).all(), "A variable has no valid training values."
mean = total / count
variance = np.maximum(squares / count - mean**2, 0)
std = np.sqrt(variance)
std = np.where(std < 1e-6, 1.0, std)

print("\nPilot training normalization:")
display(pd.DataFrame({
    "variable": VARIABLES + ["log1p_streamflow"],
    "valid_values": count,
    "mean": mean,
    "std": std,
}))

# --------------------------------------------------
# 5. Forecasting dataset, including month boundaries.
# --------------------------------------------------
class PilotForecastDataset(Dataset):
    def __init__(self, reader, mean, std):
        self.reader = reader
        self.mean = mean
        self.std = std
        self.samples = len(reader.times) - HISTORY - HORIZON + 1
        assert self.samples > 0

    def __len__(self):
        return self.samples

    def __getitem__(self, index):
        if not 0 <= index < len(self):
            raise IndexError(index)

        issue = index + HISTORY

        # Historical weather only; no future weather enters the input.
        weather, mask, _, _ = self.reader.window(index, issue)
        _, _, flow, target_mask = self.reader.window(
            issue, issue + HORIZON
        )

        safe_weather = np.where(mask, weather, self.mean[:8])
        normalized_weather = (
            (safe_weather - self.mean[:8]) / self.std[:8]
        ).astype(np.float32)
        normalized_weather[~mask] = 0.0

        inputs = np.concatenate(
            [normalized_weather, mask.astype(np.float32)], axis=-1
        )

        safe_flow = np.where(target_mask, flow, 0.0)
        targets = (
            (np.log1p(safe_flow.astype(np.float64)) - self.mean[8])
            / self.std[8]
        ).astype(np.float32)
        targets[~target_mask] = 0.0

        return {
            "x": torch.from_numpy(inputs),
            "y": torch.from_numpy(targets[..., None]),
            "target_mask": torch.from_numpy(target_mask[..., None]),
            "issue_index": issue,
        }

dataset = PilotForecastDataset(reader, mean, std)

# Explicitly test a history window crossing January–February.
issue_time = pd.Timestamp("2018-02-01 06:00")
issue_index = reader.times.get_loc(issue_time)
sample_index = issue_index - HISTORY
assert 0 <= sample_index < len(dataset)

batch = next(iter(DataLoader(
    torch.utils.data.Subset(
        dataset, [sample_index, sample_index + 1]
    ),
    batch_size=2,
    num_workers=0,   # Start with one reader for this compatibility check.
)))

# --------------------------------------------------
# 6. Existing model: real forward and backward pass.
# --------------------------------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

adjacency = np.eye(N, dtype=np.float32)
adjacency[source, target] = 1.0
supports = [
    torch.as_tensor(
        np.asarray(a), dtype=torch.float32, device=device
    )
    for a in load_adj(adjacency, "doubletransition")
]

model = GWNet(
    device=device,
    num_nodes=N,
    supports=supports,
    addaptadj=False,
    in_dim=16,
    out_dim=40,
    blocks=4,
    layers=5,
    residual_channels=8,
    dilation_channels=8,
    skip_channels=32,
    end_channels=64,
).to(device)

x = batch["x"].to(device)
y = batch["y"].to(device)
target_mask = batch["target_mask"].to(device)

assert x.shape == (2, HISTORY, N, 16)
assert y.shape == (2, HORIZON, N, 1)
assert torch.isfinite(x).all() and torch.isfinite(y).all()
assert target_mask.any()

model.train()
model.zero_grad(set_to_none=True)
output = model(x)
assert output.shape == (2, 1, N, HORIZON), output.shape

forecast = output[:, 0].permute(0, 2, 1).unsqueeze(-1)
assert forecast.shape == y.shape
assert torch.isfinite(forecast).all()

loss = (forecast[target_mask] - y[target_mask]).square().mean()
assert torch.isfinite(loss)
loss.backward()

gradients = [p.grad for p in model.parameters() if p.grad is not None]
assert gradients and all(torch.isfinite(g).all() for g in gradients)

print("\nREAL-DATA COMPATIBILITY CHECK PASSED")
print("Device:", device)
print("Six-hour intervals:", len(reader.times))
print("Candidate windows:", len(dataset))
print("First batch issue time:", issue_time)
print("First history interval:", reader.times[sample_index])
print("Last history interval:", reader.times[issue_index - 1])
print("Forecast interval range:",
      reader.times[issue_index],
      "through",
      reader.times[issue_index + HORIZON - 1])
print("Input shape:", tuple(x.shape))
print("Forecast shape:", tuple(forecast.shape))
print("Available targets:", int(target_mask.sum().item()))
print("Loss before training:", float(loss.detach().cpu()))
print("No optimizer step performed.")


Pilot training normalization:


,variable,valid_values,mean,std
0,APCP_surface,58644,2.036087,6.221892
1,TMP_2maboveground,58644,283.391614,8.933304
2,SPFH_2maboveground,58644,0.007670,0.004439
3,PRES_surface,58644,90664.449656,1713.255515
4,DSWRF_surface,58644,196.786277,224.166569
5,DLWRF_surface,58644,295.732912,62.701466
6,UGRD_10maboveground,58644,0.577614,1.368336
7,VGRD_10maboveground,58644,-0.147694,1.501174
8,log1p_streamflow,58644,0.587871,0.654061



REAL-DATA COMPATIBILITY CHECK PASSED
Device: cuda
Six-hour intervals: 724
Candidate windows: 565
First batch issue time: 2018-02-01 06:00:00
First history interval: 2018-01-02 06:00:00
Last history interval: 2018-02-01 00:00:00
Forecast interval range: 2018-02-01 06:00:00 through 2018-02-11 00:00:00
Input shape: (2, 120, 81, 16)
Forecast shape: (2, 40, 81, 1)
Available targets: 6480
Loss before training: 0.9778867959976196
No optimizer step performed.


In [4]:
from pathlib import Path
from collections import defaultdict, deque
import json

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from IPython.display import display

ROOT = Path("/data/tva_data")
PILOT = ROOT / "processed/pilot_graph/rosn7"

nodes = pd.read_parquet(PILOT / "nodes.parquet")
edges = pd.read_parquet(PILOT / "edges.parquet")
report = json.loads((PILOT / "pilot_graph_report.json").read_text())

OUTLET = int(report["outlet_comid"])
selected = set(nodes.comid.astype(int))

# --------------------------------------------------
# 1. Check directed graph structure.
# --------------------------------------------------
pairs = [
    (int(a), int(b))
    for a, b in edges[
        ["fromcomid", "tocomid"]
    ].itertuples(index=False, name=None)
]
assert OUTLET in selected
assert all(a in selected and b in selected for a, b in pairs)

upstream = defaultdict(list)
downstream = defaultdict(list)
indegree = {node: 0 for node in selected}

for a, b in set(pairs):
    downstream[a].append(b)
    upstream[b].append(a)
    indegree[b] += 1

# Every pilot node should have a downstream path to the outlet.
reaches_outlet = {OUTLET}
queue = deque([OUTLET])
while queue:
    for node in upstream[queue.popleft()]:
        if node not in reaches_outlet:
            reaches_outlet.add(node)
            queue.append(node)

# Topological traversal detects directed cycles.
degree = indegree.copy()
queue = deque(node for node, value in degree.items() if value == 0)
visited = 0
while queue:
    node = queue.popleft()
    visited += 1
    for receiver in downstream[node]:
        degree[receiver] -= 1
        if degree[receiver] == 0:
            queue.append(receiver)

checks = {
    "nodes": len(selected),
    "edges": len(pairs),
    "duplicate_edges": len(pairs) - len(set(pairs)),
    "self_edges": sum(a == b for a, b in pairs),
    "directed_acyclic": visited == len(selected),
    "nodes_reaching_outlet": len(reaches_outlet),
    "all_nodes_reach_outlet": reaches_outlet == selected,
    "internal_sink_comids": sorted(
        node for node in selected if not downstream[node]
    ),
}

# Check against preserved NHD connections, including reaches
# that may have been excluded from the candidate graph.
connectivity_path = (
    ROOT / "metadata/tennessee_nhd_connectivity.parquet"
)
if connectivity_path.exists():
    connectivity = pd.read_parquet(connectivity_path)

    incoming = connectivity.loc[
        connectivity.tocomid.isin(selected)
        & ~connectivity.fromcomid.isin(selected)
        & connectivity.fromcomid.gt(0)
    ]
    outgoing = connectivity.loc[
        connectivity.fromcomid.isin(selected)
        & ~connectivity.tocomid.isin(selected)
        & connectivity.tocomid.gt(0)
    ]

    checks["incoming_NHD_boundary_edges"] = len(incoming)
    checks["outgoing_NHD_boundary_edges"] = len(outgoing)

    print("Incoming NHD boundary edges:")
    display(incoming[["fromcomid", "tocomid"]])

    print("Outgoing NHD boundary edges:")
    display(outgoing[["fromcomid", "tocomid"]])
else:
    checks["NHD_boundary_check"] = "Connectivity file unavailable"

print("Graph checks:")
print(json.dumps(checks, indent=2))

# --------------------------------------------------
# 2. Inspect gauge proximity to nearby flowlines.
# Coordinates from the supplied TVA/CAMELS-H crosswalk.
# --------------------------------------------------
GAUGE_LAT = 35.14333333
GAUGE_LON = -82.82472222

flow_path = ROOT / "metadata/tennessee_flowlines.parquet"
flowlines = gpd.read_parquet(flow_path)
flowlines.columns = [str(c).lower() for c in flowlines.columns]
assert flowlines.crs is not None

# Limit geometry calculations to the gauge neighborhood.
geographic = flowlines.to_crs("EPSG:4326")
nearby = geographic.cx[
    GAUGE_LON - 0.05:GAUGE_LON + 0.05,
    GAUGE_LAT - 0.05:GAUGE_LAT + 0.05,
].copy()

assert not nearby.empty, "No nearby flowlines found."
assert OUTLET in set(nearby.comid), (
    "Candidate outlet is outside the gauge neighborhood."
)

projected = nearby.to_crs("EPSG:32617")
gauge = gpd.GeoSeries(
    [Point(GAUGE_LON, GAUGE_LAT)],
    crs="EPSG:4326",
).to_crs(projected.crs).iloc[0]

nearby["gauge_to_flowline_m"] = projected.geometry.distance(gauge).values
nearby["candidate_outlet"] = nearby.comid.eq(OUTLET)
nearby["in_pilot"] = nearby.comid.isin(selected)

columns = [
    c for c in [
        "comid", "gnis_name", "ftype", "totdasqkm",
        "gauge_to_flowline_m", "candidate_outlet", "in_pilot"
    ] if c in nearby.columns
]

print("\nTen nearest reaches to the ROSN7 gauge:")
display(
    nearby.sort_values("gauge_to_flowline_m")[columns].head(10)
)

print("Candidate outlet:")
display(nearby.loc[nearby.candidate_outlet, columns])

# --------------------------------------------------
# 3. Locate existing routing metadata.
# Finding a file does not establish its NWM version.
# --------------------------------------------------
routing_files = []
for directory in [ROOT / "raw/geography", ROOT / "metadata"]:
    if directory.exists():
        for path in directory.rglob("*"):
            if (
                path.is_file()
                and path.suffix.lower() in {".nc", ".csv", ".parquet"}
                and "routelink" in path.name.lower().replace("_", "").replace("-", "")
            ):
                routing_files.append(str(path))

print("\nPossible NWM routing files:")
print(json.dumps(sorted(set(routing_files)), indent=2))

Incoming NHD boundary edges:


,fromcomid,tocomid


Outgoing NHD boundary edges:


,fromcomid,tocomid
3471,22165090,22165088


Graph checks:
{
  "nodes": 81,
  "edges": 80,
  "duplicate_edges": 0,
  "self_edges": 0,
  "directed_acyclic": true,
  "nodes_reaching_outlet": 81,
  "all_nodes_reach_outlet": true,
  "internal_sink_comids": [
    22165090
  ],
  "incoming_NHD_boundary_edges": 0,
  "outgoing_NHD_boundary_edges": 1
}

Ten nearest reaches to the ROSN7 gauge:


,comid,gnis_name,ftype,totdasqkm,gauge_to_flowline_m,candidate_outlet,in_pilot
5755,22165090,Browns Mill Creek,StreamRiver,178.5087,6.908596,True,True
7208,22165148,Blue Ridge Branch,StreamRiver,3.1608,1075.952879,False,False
5754,22165088,None,StreamRiver,181.8765,1075.952879,False,False
5753,22165084,French Broad River,StreamRiver,264.3822,1205.718651,False,False
5767,22165076,Browns Mill Creek,StreamRiver,76.7259,1286.033904,False,True
5756,22165060,North Fork French Broad River,StreamRiver,97.9335,1286.033904,False,True
6115,22165086,French Broad River,StreamRiver,196.0569,1311.281841,False,False
5779,22165158,French Broad River,StreamRiver,14.1399,1311.281841,False,False
5782,22165340,East Fork French Broad River,StreamRiver,66.9897,1457.265847,False,False
7163,22165026,Galloway Creek,StreamRiver,1.6821,1538.527152,False,False


Candidate outlet:


,comid,gnis_name,ftype,totdasqkm,gauge_to_flowline_m,candidate_outlet,in_pilot
5755,22165090,Browns Mill Creek,StreamRiver,178.5087,6.908596,True,True



Possible NWM routing files:
[]
